# 📘 Домашнє завдання №21. Введення до нейромереж


##  Мета
Побудувати нейронну мережу в PyTorch для задачі бінарної класифікації:
- 1 → пасажир вижив
- 0 → пасажир не вижив

---

##  Датасет
Використати датасет **Titanic** (наприклад `titanic.csv` або Kaggle Titanic dataset).

---

##  1. Попередня обробка даних

Необхідно:

- Вибрати ознаки:
  - Pclass
  - Sex
  - Age
  - Fare
  - (за бажанням: SibSp, Parch)

- Обробити пропущені значення:
  - заповнити або видалити

- Перетворити категоріальні змінні:
  - `Sex` → 0 / 1

- Нормалізувати числові ознаки:
  - використати `StandardScaler`

⚠️ Важливо:
- scaler fit робити **тільки на train set**
- test set тільки transform

---

## 2. Розбиття даних

Розділити датасет на:
- train (80%)
- test (20%)

---

## 3. Побудова моделі (PyTorch)

Створити нейронну мережу:

- Input layer: кількість ознак
- 1–2 hidden layers
- Activation function: ReLU
- Output layer: 1 нейрон
- Activation: Sigmoid

---

## 4. Навчання моделі

Використати:

- Loss function:
  - `BCELoss`

- Optimizer:
  - `Adam`

- epochs: 50–150 (на вибір)

Під час навчання:
- виводити loss кожні 10 епох

---

## 5. Оцінка моделі

Порахувати:

- Accuracy на test set
- (опційно) confusion matrix

---

## 6. Експерименти

Спробувати:

- змінити кількість нейронів у hidden layers
- змінити learning rate
- додати ще один hidden layer

---

## Результати (що здати)

Потрібно надати:

- код навчання моделі
- фінальний accuracy
- короткий висновок:
  - що вплинуло на результат
  - яка конфігурація дала найкращий результат

In [4]:
# Silent installation or update

# Clean cache
!python3 -m pip cache purge -q

# Force updating
package_update = [
    "pip",
    "scikit-learn",
    "pandas",
    "kagglehub",
    "kagglesdk",
]

for package_name in package_update:
    !bash -c "python3 -m pip install -U '{package_name}' -q"

# Install missing packages
package_array = [
    "jinja2",
    "ipywidgets",
    "nbformat",
    "kagglehub[pandas-datasets]",
    "numpy",
    "matplotlib",
    "scipy",
    "statsmodels",
    "joblib",
    "seaborn",
]

for package_name in package_array:
    !bash -c "python3 -m pip show '{package_name}' > /dev/null 2>&1 || python3 -m pip install -U '{package_name}' -q"


In [5]:
# # Synchronization with remote source
#
# import shutil
# from pathlib import Path
#
# # Input data
hm_version = 21
#
# # Solution
# git_project_url = f"https://github.com/BogdanPinchuk/DataScience-PBY_HW{hm_version}.git"
# main_file_name = f"Bohdan_Pinchuk_DS_HW{hm_version}.ipynb"
#
# # upload all files
# current_path = !pwd
# current_path = current_path[0]
# parent_path = !dirname "$current_path"
# parent_path = parent_path[0]
# temp_path = f"{parent_path}/temp"
#
# # Clone data
# !rm -rf "$temp_path"
# !git clone "$git_project_url" "$temp_path"
#
# source = Path(temp_path)
# destination = Path(current_path)
# exclude = {main_file_name, ".git", ".idea"}
#
# for item in source.iterdir():
#     if item.name in exclude:
#         continue
#
#     target = destination / item.name
#     if item.is_dir():
#         shutil.copytree(item, target, dirs_exist_ok=True)
#     else:
#         shutil.copy2(item, target)
#
# # Clean temp folder
# !rm -rf "$temp_path"

## ✳️ Підготовка датасетів

In [6]:
# Downloading data

import pandas as pd
import apps.main as mn

# Input data
update_db = False
db_file_name = f"resources/store_hw{hm_version}.db"

# Solution
pd.options.display.float_format = "{:g}".format  # type: ignore
titanic_dataset = mn.download_and_extract_from_kagglehub("yasserh/titanic-dataset",
                                                         "Titanic-Dataset.csv",
                                                         db_file_name,
                                                         update_db=update_db)

# Print result
display(titanic_dataset)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22,1,0,A/5 21171,7.25,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26,0,0,STON/O2. 3101282,7.925,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35,1,0,113803,53.1,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35,0,0,373450,8.05,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27,0,0,211536,13,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19,0,0,112053,30,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.45,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26,0,0,111369,30,C148,C
